# Selective-Layer LoRA — Colab runner (Hashcode 14.0)
Runtime → Change runtime type → **T4 GPU**. Set `QUEUE` to your letter. Run all cells top to bottom.

One-time: add a Colab secret `GH_TOKEN` (fine-grained GitHub token, *Contents: read & write* on the repo) via the 🔑 icon on the left.

In [ ]:
QUEUE = 'A'            # M1 = 'A', M2 = 'B', M3 = 'C'
REPO  = 'YOUR_GH_USER/selective-layer-lora'   # <- edit once
WHO   = 'M1'           # used in commit author name
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/hashcode_sllora'   # shared folder (shortcut added to My Drive)
import os; os.makedirs(f'{WORK}/outputs', exist_ok=True); os.makedirs(f'{WORK}/results', exist_ok=True)

In [ ]:
TOKEN = userdata.get('GH_TOKEN')
!rm -rf /content/repo && git clone -q https://{TOKEN}@github.com/{REPO}.git /content/repo
%cd /content/repo
!git config user.name "{WHO}-colab" && git config user.email "{WHO}@frozenlayers.dev"
!pip uninstall -y -q torchao   # Colab's old torchao breaks new peft; we don't use it
!pip install -q -r requirements.txt
!python tests/smoke_test.py 2>&1 | tail -3

### Shared data split (identical on every account)
The first account to run this writes `data/splits.json` and copies it to Drive; the others reuse it.

In [ ]:
import shutil
if os.path.exists(f'{WORK}/splits.json'):
    os.makedirs('data', exist_ok=True); shutil.copy(f'{WORK}/splits.json', 'data/splits.json')
else:
    !python -c "from sllora.data import build_splits; build_splits(cache_path='data/splits.json')"
    shutil.copy('data/splits.json', f'{WORK}/splits.json')
!git pull -q   # get latest scores / cost model from teammates

### M3 only, first thing: memory probes + importance scores (≈10 min)

In [ ]:
if QUEUE == 'C':
    !python scripts/probe_memory.py
    !python scripts/score_layers.py --drift --importance
    !git add -f results/scores && git commit -qm "scores: memory probes + cost model + importance" && git pull -q --rebase && git push -q

### Run my queue (resumable: re-run this cell after any disconnect, finished runs are skipped)

In [ ]:
!git pull -q
!python scripts/run_sweep.py --queue {QUEUE} --out {WORK}/outputs --drive_results {WORK}/results --git_push

### Aggregate + plots (anyone, any time)

In [ ]:
!git pull -q && python scripts/aggregate.py && python scripts/plot.py
from IPython.display import Image, display
for p in ['levers','pareto','layer_scores','probes']:
    f = f'results/plots/{p}.png'
    if os.path.exists(f): display(Image(f))

### Demo (M3, for reviews)

In [ ]:
%env SLLORA_OUT={WORK}/outputs
!python app/demo.py